In [0]:
from pyspark.sql.functions import when, col,concat_ws, substring, regexp_replace, sha2, concat, upper, lit

In [0]:
df = spark.read.csv(
    "s3://s3-lge-he-inbound-kic-dev/HEDS/HEDS-1391/추출대상_MAC_List_LQC_분석.csv",
    header=True,
    inferSchema=True
)
display(df)

In [0]:
df = df.withColumn("mac_addr", upper(df["MAC_ADDRESS"]))
display(df)

In [0]:
df = df.withColumn(
    "mac_addr",
    concat_ws(":",
        substring(col("mac_addr"), 1, 2),
        substring(col("mac_addr"), 3, 2),
        substring(col("mac_addr"), 5, 2),
        substring(col("mac_addr"), 7, 2),
        substring(col("mac_addr"), 9, 2),
        substring(col("mac_addr"), 11, 2),
    )
)

display(df)

In [0]:
tv_salt = dbutils.secrets.get("admin", "salt")

df=df.withColumn("mac_addr_hashed", when(col("mac_addr").isNull() | (col("mac_addr") == ''), col("mac_addr")).otherwise(sha2(concat(col("mac_addr"), lit(tv_salt)), 256)))

display(df)

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable(f"sandbox.z_luckyminki_choi.heds_1391")